<a href="https://colab.research.google.com/github/asennakesavan/InceptezGenAI-Batch26/blob/main/Customer_Campaign_response_prediction_model1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer #Impute the missing values with MEAN / MEDIAN / MODE
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,precision_score, recall_score, f1_score, confusion_matrix)

In [8]:
train_df=pd.read_csv("train.csv")
test_df=pd.read_csv("test.csv")

In [9]:
print(f"shape of train dataset is ", {train_df.shape})
print(f"shape of train dataset is ", {test_df.shape})

shape of train dataset is  {(36168, 17)}
shape of train dataset is  {(9043, 17)}


In [10]:
df = pd.concat([train_df,test_df], ignore_index=True)


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          36168 non-null  object
 17  id         9043 non-null   object
dtypes: int64(7), object(11)
memory usage: 6.2+ MB


In [12]:
df.isna().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


In [13]:
df['y'].value_counts()

,count
y,
no,31937
yes,4231


In [14]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,id
0,37,admin.,single,secondary,no,483,no,no,cellular,31,jul,11,11,-1,0,unknown,no,NaN
1,53,management,divorced,tertiary,no,253,no,no,cellular,27,aug,202,6,-1,0,unknown,no,NaN
2,55,management,married,unknown,no,1504,yes,no,cellular,31,aug,186,3,101,2,success,yes,NaN
3,33,services,married,secondary,no,0,yes,no,cellular,14,jul,163,2,-1,0,unknown,no,NaN
4,27,admin.,single,primary,no,528,yes,yes,cellular,18,may,267,2,311,2,failure,no,NaN


In [15]:
train_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,37,admin.,single,secondary,no,483,no,no,cellular,31,jul,11,11,-1,0,unknown,no
1,53,management,divorced,tertiary,no,253,no,no,cellular,27,aug,202,6,-1,0,unknown,no
2,55,management,married,unknown,no,1504,yes,no,cellular,31,aug,186,3,101,2,success,yes
3,33,services,married,secondary,no,0,yes,no,cellular,14,jul,163,2,-1,0,unknown,no
4,27,admin.,single,primary,no,528,yes,yes,cellular,18,may,267,2,311,2,failure,no


In [16]:
train_df['y'].value_counts()

,count
y,
no,31937
yes,4231


In [17]:
train_df['poutcome'].value_counts()

,count
poutcome,
unknown,29549
failure,3952
other,1460
success,1207


In [23]:
train_Categorical_feature = train_df.select_dtypes(include=['object']).columns.drop('y')

In [24]:
train_Numerical_feature = train_df.select_dtypes(include=['int64']).columns.drop('duration')

In [32]:
test_Categorical_feature = test_df.select_dtypes(include=['object'])
test_Numerical_feature = test_df.select_dtypes(include=['int64']).columns.drop('duration')

In [25]:
train_feature = train_df.select_dtypes(include=['object','int64']).columns.drop(['y','duration'])

In [35]:
test_feature = test_df.select_dtypes(include=['object','int64']).columns.drop(['duration'])

In [43]:
target_column ='y'

In [45]:
y_train=train_df[target_column]

In [26]:
train_Numerical_feature

Index(['age', 'balance', 'day', 'campaign', 'pdays', 'previous'], dtype='object')

In [27]:
train_Categorical_feature

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome'],
      dtype='object')

In [28]:
numeric_transformer = Pipeline(steps=[ #numerical columns
    # ('imputer', SimpleImputer(strategy='median')), #remove null - Median
    ('scaler', StandardScaler()) #0-1
])

In [29]:
categorical_transformer = Pipeline(steps=[ #categorical columns
    # ('imputer', SimpleImputer(strategy='most_frequent')), #remove null - MODE
    ('onehot', OneHotEncoder(handle_unknown='ignore')) #OneHotEncoding
])

In [64]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, train_Numerical_feature),
        ('cat', categorical_transformer, train_Categorical_feature)
    ])

knn = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', KNeighborsClassifier(n_neighbors=5))])
logistic_regression = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])
Decision_Tree = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier',DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42, class_weight='balanced'))])

In [33]:
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numeric_transformer, test_Numerical_feature),
#         # ('cat', categorical_transformer, test_Categorical_feature)
#     ])

In [65]:
preprocessor.fit(train_df[train_feature])
train_preprocessed = preprocessor.transform(train_df[train_feature])
test_preprocessed = preprocessor.transform(test_df[test_feature])

In [69]:
# Logistic Regression
x_train = train_df[train_feature]
x_test= test_df[test_feature]
logistic_regression.fit(x_train,y_train)
y_Log_pred_train = logistic_regression.predict(x_train)
y_Log_pred_test = logistic_regression.predict(x_test)

print("Confusion Matrix")
print(confusion_matrix(y_train, y_Log_pred_train))
print("Accuracy:", accuracy_score(y_train, y_Log_pred_train))
print("Precision:", precision_score(y_train, y_Log_pred_train, average='macro'))
print("Recall:", recall_score(y_train, y_Log_pred_train, average='macro'))
print("F1 Score:", f1_score(y_train, y_Log_pred_train,  average='macro'))

Confusion Matrix
[[24578  7359]
 [ 1566  2665]]
Accuracy: 0.753234903782349
Precision: 0.6029814552784456
Recall: 0.6997261700085254
F1 Score: 0.610119591870546


In [70]:
pd.Series(y_Log_pred_test).value_counts()

,count
no,6603
yes,2440


In [79]:
# KNN Regression
knn.fit(x_train,y_train)
y_KNN_pred_train = knn.predict(x_train)
y_KNN_pred_test = knn.predict(x_test)

print("Confusion Matrix")
print(confusion_matrix(y_train, y_KNN_pred_train))
print("Accuracy:", accuracy_score(y_train, y_KNN_pred_train))
print("Precision:", precision_score(y_train, y_KNN_pred_train, average='macro'))
print("Recall:", recall_score(y_train, y_KNN_pred_train, average='macro'))
print("F1 Score:", f1_score(y_train, y_KNN_pred_train,  average='macro'))

Confusion Matrix
[[31432   505]
 [ 2932  1299]]
Accuracy: 0.9049712452997124
Precision: 0.8173723352004792
Recall: 0.6456036182437199
F1 Score: 0.6893247382356185


In [72]:
pd.Series(y_KNN_pred_test).value_counts()

,count
no,8644
yes,399


In [83]:
knn_predictions_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': y_KNN_pred_test
})

knn_predictions_df.to_csv("Sample_submission.csv")

In [75]:
# Decision Tree Regression
Decision_Tree.fit(x_train,y_train)
y_DT_pred_train = Decision_Tree.predict(x_train)
y_DT_pred_test = Decision_Tree.predict(x_test)

print("Confusion Matrix")
print(confusion_matrix(y_train, y_DT_pred_train))
print("Accuracy:", accuracy_score(y_train, y_DT_pred_train))
print("Precision:", precision_score(y_train, y_DT_pred_train, average='macro'))
print("Recall:", recall_score(y_train, y_DT_pred_train, average='macro'))
print("F1 Score:", f1_score(y_train, y_DT_pred_train,  average='macro'))

Confusion Matrix
[[24727  7210]
 [ 1807  2424]]
Accuracy: 0.7506912187569121
Precision: 0.5917537906054595
Recall: 0.6735786228333439
F1 Score: 0.5977221055203235


In [76]:
pd.Series(y_DT_pred_test).value_counts()

,count
no,6663
yes,2380
